In [ ]:
%reload_ext autoreload
%autoreload 2

from pathlib import Path
import subprocess
from PIL import Image

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

import flexiznam as flz
from cottage_analysis.analysis import common_utils
from cottage_analysis.pipelines import pipeline_utils
from cottage_analysis.plotting import rsof_plots, depth_selectivity_plots
from cottage_analysis.plotting import style
from v1_depth_map.figure_utils.treadmill import (
    load_treadmill_population_neurons_df,
    compute_treadmill_rsof_bins,
    add_trial_average_rsof_columns,
    plot_treadmill_stim_sampling,
    plot_treadmill_protocol,
)
from v1_depth_map.figure_utils.rsof_integration import (
    plot_angle_eccentricity_polar,
    plot_g2d_fit_schematic,
    plot_expected_depth_vs_treadmill,
    SEMIMAJOR_COLOR,
    SEMIMINOR_COLOR,
)


In [ ]:
# Register the manuscript font faces (Arial regular + bold + italic, Arial Narrow) and
# apply the publication rcParams: vector fonttypes, font sizes, tick/label padding.
# `style.savefig` then expands the SVG `font:` shorthand so Illustrator reads the
# family, size and weight correctly - see cottage_analysis.plotting.style for both.
from cottage_analysis.plotting import style
from cottage_analysis.plotting.style import CM, FONTSIZE_DICT

style.setup_figure_fonts()

In [ ]:
project = "colasa_3d-vision_revisions"
flexilims_session = flz.get_flexilims_session(project)
from v1_depth_map.paths import get_figures_roots

READ_ROOT, SAVE_ROOT = get_figures_roots(flexilims_session)
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

regenerate_cached_files = False  # set to True to regenerate_cached_files everything

In [ ]:
# Load the treadmill population data (all sessions with the treadmill protocol), used
# to look up example cells by roi_uid and populate the population panels (F, K, L).
(
    neurons_df_treadmill,
    simul_df_treadmill_population,
    simul_df_spheres_population,
    valid_treadmill_sessions,
    treadmill_sessions,
) = load_treadmill_population_neurons_df(flexilims_session, load_simulated=False)


In [ ]:
# Trial-averaged treadmill g2d fits (one sample per trial rather than per imaging frame):
# derive the tuning-ellipse geometry and the empirical-null significance flags used by the
# population panel below. `_plateau` matches the `method="plateau"` onset detection used
# when loading the example session's trials.
TA = "_treadmill_trial_average_plateau"
MIN_SIGMA = add_trial_average_rsof_columns(
    neurons_df_treadmill, ta_suffix=TA, null_method="empirical"
)

# Population for the polar panel: significant 2D-Gaussian fit on the treadmill, among
# neurons whose depth tuning was established in CLOSED LOOP (`is_depth_neuron`, no
# suffix). The two criteria deliberately come from different protocols: depth tuning
# from closed loop, where the animal's own movement varies depth, and the RS/OF
# geometry from the treadmill. No elongation cut here - elongation is the radial axis,
# so "ridge" fits (unbounded along one axis) simply land far out rather than being hidden.
ndf_polar = neurons_df_treadmill[
    neurons_df_treadmill[f"rsof_test_rsq_closedloop_g2d{TA}_sig"]
    & neurons_df_treadmill["is_depth_neuron"].fillna(False)
].dropna(subset=[f"g2d_theta{TA}", f"g2d_semimajor{TA}", f"g2d_semiminor{TA}"])

# A near-circular tuning ellipse has no meaningful orientation, so only clearly
# elongated fits enter the orientation histogram and the von Mises mixture. They stay
# on the polar scatter (drawn in black there) rather than being dropped silently. The
# cut is on the elongation (sigma_major / sigma_minor), the same quantity the polar
# panel uses as its radial axis; 1.4:1 corresponds to an eccentricity of 0.70.
#
# 1.4:1 is where the orientation stops being reproducible, rather than a round number: the
# last cell of this notebook estimates a per-neuron SE on theta from the five
# cross-validation fold fits (`rsof_train_popt_*`, a delete-a-group jackknife), and the
# local median of that SE crosses 22.5 deg - half the 45 deg between the ratio-tuned
# (45 deg) and cardinal (0/90 deg) hypotheses, i.e. the resolution needed to tell them
# apart - at 1.4:1.
ELONGATION_CUTOFF = 1.4
elongation_polar = ndf_polar[f"g2d_semimajor{TA}"].astype(float) / ndf_polar[
    f"g2d_semiminor{TA}"
].astype(float)
elong_ok = elongation_polar > ELONGATION_CUTOFF

print(
    f"\n{len(ndf_polar)} neurons in the polar panel "
    f"({len(ndf_polar) / len(neurons_df_treadmill):.1%} of {len(neurons_df_treadmill)}), "
    f"{ndf_polar.session.nunique()} sessions"
)
print(
    f"{int(elong_ok.sum())} with elongation > {ELONGATION_CUTOFF} "
    f"(orientation histogram/fit), {int((~elong_ok).sum())} below the cut"
)
print(ndf_polar.groupby("session").size().rename("n_neurons"))

In [ ]:
# Load the example session's treadmill (motorised wheel) trials, used for the
# treadmill protocol trace, stimulus sampling, and example-cell RS/OF tuning panels.
EXAMPLE_SESSION = "PZAG17.3a_S20250402"
example_mouse, example_session = EXAMPLE_SESSION.split("_")

ndf, trials_df_tm, trials_df_sphere = pipeline_utils.load_treadmill_and_sphere_datasets(
    project,
    example_mouse,
    example_session,
    photodiode_protocol=5,
    filter_datasets={"anatomical_only": 3, "annotated": True},
    recording_type="two_photon",
    protocol_base_sphere="SpheresPermTubeReward",
    tread_kwargs=dict(method="plateau"),
)

# Same trials without cutting the acceleration ramp (protocol figure)
_, trials_df_tm_no_cut, _ = pipeline_utils.load_treadmill_and_sphere_datasets(
    project,
    example_mouse,
    example_session,
    photodiode_protocol=5,
    filter_datasets={"anatomical_only": 3, "annotated": True},
    recording_type="two_photon",
    protocol_base_sphere="SpheresPermTubeReward",
    tread_kwargs=dict(cut_trial_end=None, trial_duration=None, acceleration_time=None),
)

# Imaging frame rate
suite2p_ds = flz.get_datasets(
    origin_name=EXAMPLE_SESSION,
    dataset_type="suite2p_rois",
    filter_datasets=dict(annotated=True),
    flexilims_session=flexilims_session,
    allow_multiple=False,
)
fs_treadmill = suite2p_ds.extra_attributes["fs"]

max_abs_rs2motor_diff_ratio = 0.3
rs_bins, of_bins, tick_dict = compute_treadmill_rsof_bins(trials_df_tm)


In [ ]:
# How many components does the orientation distribution need? Fit the axial von Mises
# mixture for k = 1 .. MAX_K and compare BIC. Orientation is circular with 180 deg
# periodicity, so the mixture is fit on the doubled angle (see
# v1_depth_map.figure_utils.von_mises). Same data (elongated fits only, `elong_ok`), k
# range and seed as the figure cell below, so the k picked here is the one drawn there.
from v1_depth_map.figure_utils import von_mises as vm

MAX_K_SELECT = 5
ks = list(range(1, MAX_K_SELECT + 1))
angles_select = (
    ndf_polar.loc[elong_ok, f"g2d_theta{TA}"].astype(float).dropna().to_numpy()
)


def wrap_axial_deg(deg):
    """Wrap an axial angle to [-45, 135), matching fit_gb.get_gaussian_angle."""
    return (np.asarray(deg) + 45) % 180 - 45


vm_sel = vm.model_selection(
    np.radians(angles_select), max_k=MAX_K_SELECT, seed=42, verbose=False
)
vm_bic = {k: vm_sel[k]["bic"] for k in ks}
k_vm_best = min(vm_bic, key=vm_bic.get)

fig_bic, ax_bic = plt.subplots(1, 1, figsize=(3.8, 2.6))
ax_bic.plot(ks, [vm_bic[k] for k in ks], "o-", color="#CC79A7")
ax_bic.axvline(k_vm_best, color="k", ls=":", lw=1)
ax_bic.set_xticks(ks)
ax_bic.set_xlabel("n components")
ax_bic.set_ylabel("BIC")
ax_bic.set_title(f"von Mises mixture: best k = {k_vm_best}", fontsize=9)
ax_bic.spines[["top", "right"]].set_visible(False)
fig_bic.tight_layout()

print(
    f"n = {len(angles_select)} neurons (elongation > {ELONGATION_CUTOFF}), "
    f"k = {ks[0]} .. {ks[-1]}\n"
)

print("Axial von Mises mixture (fit on the doubled angle)")
for k in ks:
    print(f"  k={k}  BIC={vm_bic[k]:10.1f}" + ("   <-- best" if k == k_vm_best else ""))
r_best = vm_sel[k_vm_best]
# mu comes back on the doubled-angle circle, so it can land outside [-45, 135).
mu_deg = wrap_axial_deg(np.degrees(r_best["mu_k"]))
print(f"  components of best k={k_vm_best}:")
for i in np.argsort(mu_deg):
    print(
        f"    mu={mu_deg[i]:7.1f} deg"
        f"   kappa={r_best['kappa_k'][i]:6.2f}"
        f"   fraction={r_best['pi_k'][i]:.3f}"
    )

In [ ]:
# Suggest example cells for EXAMPLE_SESSION: significant treadmill RS/OF (g2d) fit and
# a gaussian angle close to 45 deg (i.e. tuned to the RS/OF ratio, so depth-tuned on the
# treadmill). Sorted by how close the angle is to 45 deg, then by fit quality.
target_angle = 90  # degrees
angle_tol = 15  # keep cells within +/- this of target_angle

candidates = neurons_df_treadmill[
    (neurons_df_treadmill.session == EXAMPLE_SESSION)
    & neurons_df_treadmill.rsof_neuron_treadmill
    & ((neurons_df_treadmill.g2d_theta_treadmill - target_angle).abs() < angle_tol)
].copy()
candidates["angle_dist"] = (candidates.g2d_theta_treadmill - target_angle).abs()
candidates = candidates.sort_values(
    ["angle_dist", "rsof_test_rsq_closedloop_g2d_treadmill"],
    ascending=[True, False],
)

print(f"{len(candidates)} candidate cells in {EXAMPLE_SESSION}")
display(
    candidates[
        [
            "roi_uid",
            "g2d_theta_treadmill",
            "g2d_eccentricity_treadmill",
            "rsof_test_rsq_closedloop_g2d_treadmill",
            "g2d_preferred_RS_treadmill",
            "g2d_preferred_OF_treadmill",
            "is_depth_neuron",
            "is_depth_neuron_treadmill",
            "preferred_depth_closedloop_crossval_treadmill",
        ]
    ].head(20)
)

In [ ]:
# Example cells:
# Panels D, E: Free locomotion depth-tuned cell tested on motorized wheel
EXAMPLE_CELL_TREADMILL = "PZAG17.3a_S20250402_83"

# Panels G, H, I: Three exemplar tuning profiles on the motorized wheel
# RS: "PZAG17.3a_S20250402_235"
# OF: "PZAG17.3a_S20250402_309"
# Ratio / depth: "PZAG17.3a_S20250402_31"
EXAMPLE_CELL = "PZAG17.3a_S20250402_235"
EXAMPLE_CELL2 = "PZAG17.3a_S20250402_309"
EXAMPLE_CELL3 = "PZAG17.3a_S20250402_31"

# Panel J: Cell whose 2D-Gaussian fit illustrates the fit parameters in the bottom row: strongly
# fit (R2 0.71), elongated (ecc 0.75) and oriented at 43 deg, with a preferred RS/OF
# well inside the sampled grid so the whole ellipse is visible.
EXAMPLE_CELL4 = "PZAG17.3a_S20250402_142"

example_cell_treadmill = neurons_df_treadmill[
    neurons_df_treadmill.roi_uid == EXAMPLE_CELL_TREADMILL
].iloc[0]


In [ ]:
# ==============================================================================
# Assembled Figure 3 (Panels A - L)
# Full multi-panel layout matching manuscript submission (fig3.pdf)
# ==============================================================================
cm = 1 / 2.54
FIG_W_CM, FIG_H_CM = 18.28, 15.86
ASPECT = FIG_W_CM / FIG_H_CM  # x-fraction -> y-fraction, to keep panels square

fig = plt.figure(figsize=(FIG_W_CM * cm, FIG_H_CM * cm))
ax_bg = fig.add_axes([0, 0, 1, 1])
ax_bg.set_xticks([])
ax_bg.set_yticks([])
ax_bg.axis("off")

fontsize_dict = FONTSIZE_DICT

# ==============================================================================
# TOP TIER: PANELS A - F (Motorized wheel experiment and validation)
# ==============================================================================

# --- Panel A: Motorized Wheel Schematic ---
fig.text(
    0.013,
    0.985,
    "A",
    fontsize=fontsize_dict.get("panel", 10),
    fontweight="bold",
    ha="left",
    va="top",
)
schematic_path = Path("v1_depth_map/figures/schematic_fig3_motorized_wheel.png")
if not schematic_path.exists():
    schematic_path = Path("v1_depth_map/figures/schematic_fig2_motorized_wheel.png")
if not schematic_path.exists():
    schematic_path = Path(
        "/Users/blota/Documents/code/v1_depth_map/v1_depth_map/figures/schematic_fig2_motorized_wheel.png"
    )

if schematic_path.exists():
    schematic_img = Image.open(schematic_path)
    ax_a = fig.add_axes([0.015, 0.770, 0.165, 0.200])
    ax_a.imshow(schematic_img)
    ax_a.axis("off")
else:
    ax_a = fig.add_axes([0.015, 0.770, 0.165, 0.200])
    ax_a.axis("off")

# --- Panel B: Motorized Wheel Protocol Trace ---
fig.text(
    0.013,
    0.760,
    "B",
    fontsize=fontsize_dict.get("panel", 10),
    fontweight="bold",
    ha="left",
    va="top",
)
ax_b = fig.add_axes([0.025, 0.630, 0.160, 0.125])
plot_treadmill_protocol(
    trials_df=trials_df_tm,
    example_trials=np.arange(58, 62),
    fs=fs_treadmill,
    trials_df_no_cut=trials_df_tm_no_cut,
    max_abs_rs2motor_diff_ratio=max_abs_rs2motor_diff_ratio,
    ax=ax_b,
    xlim=None,
    fontsize_dict=fontsize_dict,
    plot_exclude_frames=False,
)

# --- Panel C: Treadmill Stimulus Sampling ---
fig.text(
    0.190,
    0.985,
    "C",
    fontsize=fontsize_dict.get("panel", 10),
    fontweight="bold",
    ha="left",
    va="top",
)
ax_c = fig.add_axes([0.235, 0.655, 0.140, 0.275])
plot_treadmill_stim_sampling(
    trials_df=trials_df_tm,
    ax=ax_c,
    fontsize_dict=fontsize_dict,
    markersize=2,
    trial_markersize=4,
    legend_kwargs=dict(bbox_to_anchor=(1.05, 1.25), handletextpad=0.2, borderpad=0),
    ylim=(0.5, None),
)

# --- Panel D: Free Locomotion Depth Tuning for Treadmill Example ---
fig.text(
    0.399,
    0.985,
    "D",
    fontsize=fontsize_dict.get("panel", 10),
    fontweight="bold",
    ha="left",
    va="top",
)
ax_d = fig.add_axes([0.455, 0.865, 0.140, 0.095])
depth_selectivity_plots.plot_depth_tuning_curve(
    neurons_df=ndf,
    trials_df=trials_df_sphere,
    roi=example_cell_treadmill.roi,
    ax=ax_d,
    rs_thr=0.05,
    use_col="depth_tuning_popt_closedloop_running",
    plot_fit=True,
    plot_smooth=False,
    linewidth=1.5,
    linecolor="royalblue",
    closed_loop=1,
    fontsize_dict=fontsize_dict,
    markersize=5,
    markeredgecolor="w",
)
ax_d.tick_params(axis="both", which="major", labelsize=fontsize_dict["tick"], pad=1)
ax_d.tick_params(axis="x", rotation=45, pad=1)
ax_d.xaxis.label.set_size(fontsize_dict["label"])
ax_d.yaxis.label.set_size(fontsize_dict["label"])
ax_d.set_xlabel("Depth (cm)", labelpad=1)
ax_d.set_title("Free locomotion", fontsize=fontsize_dict["title"], pad=3)
ax_d.set_ylim(-0.15, 1.35)
ax_d.set_yticks([-0.1, 1.3], labels=["-0.1", "1.3"])
sns.despine(ax=ax_d, offset=3, trim=True)

# --- Panel E: Free Locomotion vs Motorized Wheel Matrices ---
fig.text(
    0.396,
    0.805,
    "E",
    fontsize=fontsize_dict.get("panel", 10),
    fontweight="bold",
    ha="left",
    va="top",
)
motor_speeds = np.round(np.unique(trials_df_tm.MotorSpeed_stim.map(np.nanmedian)))
ms_log = np.log2(motor_speeds)
rs_bw = np.median(np.diff(ms_log))
example_rs_bins = 2 ** np.arange(
    ms_log[0] - rs_bw * 1.5, ms_log[-1] + rs_bw * 2, rs_bw
)
example_rs_bins = np.insert(example_rs_bins, 0, 0)
of_speeds = np.round(
    np.unique(trials_df_tm.expected_optic_flow_stim.map(np.nanmedian))
)
of_log = np.log2(of_speeds)
of_bw = np.median(np.diff(of_log))
example_of_bins = 2 ** np.arange(of_log[0] - of_bw * 0.5, of_log[-1] + of_bw, of_bw)
example_of_bins = np.insert(example_of_bins, 0, 0)

rs_logbin = np.log2(example_rs_bins[1:])
rs_bin_middle = np.diff(rs_logbin) / 2 + rs_logbin[:-1]
of_logbin = np.log2(example_of_bins[1:])
of_bin_middle = np.diff(of_logbin) / 2 + of_logbin[:-1]
example_tick_dict = dict(
    rs_tick_select=np.log2(np.array([2, 8, 32, 128])),
    rs_tick_values=[2, 8, 32, 128],
    of_tick_select=of_bin_middle,
    of_tick_values=[1, 4, 16, 64, 256, 1024],
)

example_range_kwargs = dict(
    log_range={"log_base": 2},
    rs_bins=example_rs_bins,
    of_bins=example_of_bins,
    tick_dict=example_tick_dict,
)
motor_range_kwargs = dict(
    example_range_kwargs,
    rs_bins=example_rs_bins[:-1],
    tick_dict=dict(
        example_tick_dict,
        rs_tick_select=example_tick_dict["rs_tick_select"][:-1],
        rs_tick_values=example_tick_dict["rs_tick_values"][:-1],
    ),
)

ax_e1 = fig.add_axes([0.530, 0.640, 0.065, 0.130])
vmin, vmax, _ = rsof_plots.plot_RS_OF_matrix(
    trials_df=trials_df_tm,
    roi=example_cell_treadmill.roi,
    is_closed_loop=1,
    max_abs_rs2motor_diff_ratio=max_abs_rs2motor_diff_ratio,
    ax=ax_e1,
    fontsize_dict=fontsize_dict,
    return_matrix=True,
    title="Motorized\nwheel",
    vmin=0,
    vmax=1,
    **motor_range_kwargs,
)

ax_e0 = fig.add_axes([0.440, 0.640, 0.080, 0.130])
vmin, vmax, _ = rsof_plots.plot_RS_OF_matrix(
    trials_df=trials_df_sphere,
    roi=example_cell_treadmill.roi,
    is_closed_loop=1,
    max_abs_rs2motor_diff_ratio=max_abs_rs2motor_diff_ratio,
    ax=ax_e0,
    fontsize_dict=fontsize_dict,
    return_matrix=True,
    title="Free\nlocomotion",
    cbar_width=None,
    vmin=vmin,
    vmax=vmax,
    **example_range_kwargs,
)
ax_e0.set_ylabel(
    "Optic flow speed (°/s)", fontsize=fontsize_dict["label"], labelpad=1
)
ax_e1.set_ylabel("")
ax_e1.set_yticks([])
for a in [ax_e0, ax_e1]:
    a.set_xlabel("")
    a.tick_params(axis="both", labelsize=fontsize_dict["tick"], pad=1)
fig.text(
    0.525,
    0.605,
    "Running speed (cm/s)",
    ha="center",
    va="bottom",
    fontsize=fontsize_dict["label"],
)

# --- Panel F: Population Treadmill Ratio vs Free Locomotion Depth ---
fig.text(
    0.629,
    0.985,
    "F",
    fontsize=fontsize_dict.get("panel", 10),
    fontweight="bold",
    ha="left",
    va="top",
)
rsof_sfx, depth_sfx = TA, ""
rsq_col = f"rsof_test_rsq_closedloop_g2d{rsof_sfx}"
rsq_vals = pd.to_numeric(neurons_df_treadmill[rsq_col], errors="coerce").dropna().values
finite_vals = rsq_vals[np.isfinite(rsq_vals) & (rsq_vals >= -1)]
thr, sigma = common_utils.empirical_null_threshold(finite_vals, percentile=95)

valid_neurons_treadmill = neurons_df_treadmill[
    neurons_df_treadmill["is_depth_neuron"]
    & (pd.to_numeric(neurons_df_treadmill[rsq_col], errors="coerce") > thr)
]

XTICKS = [0.08, 0.8, 8, 80, 800, 8000]
YTICKS = [2, 20, 200, 2000]

def _lim(data, ticks):
    return (
        min(np.log(ticks[0]), data.min()),
        max(np.log(ticks[-1]), data.max()),
    )

ax_f = fig.add_axes([0.700, 0.650, 0.270, 0.275])
sc, plotted_element = plot_expected_depth_vs_treadmill(
    ax_f,
    valid_neurons_treadmill,
    fontsize_dict,
    s=12,
    alpha=0.6,
    color="#444444",
    ec="none",
    plot_fit=False,
    plot_unity=True,
    stat_type="spearman",
    suffix=rsof_sfx,
    depth_suffix=depth_sfx,
)
_kx = plotted_element["x"][np.isfinite(plotted_element["x"])]
_ky = plotted_element["y"][np.isfinite(plotted_element["y"])]
xlim_f = _lim(_kx, XTICKS)
ylim_f = _lim(_ky, YTICKS)
ax_f.set_xlim(*xlim_f)
ax_f.set_ylim(*ylim_f)
sc.set_clip_on(False)
ax_f.set_xticks(np.log(XTICKS))
ax_f.set_xticklabels([f"{t:g}" for t in XTICKS])
ax_f.set_yticks(np.log(YTICKS))
ax_f.set_yticklabels([f"{t:g}" for t in YTICKS])
ax_f.set_ylabel(
    "Preferred depth during free locomotion (cm)", fontsize=fontsize_dict["label"]
)
ax_f.set_xlabel(
    "Ratio of preferred RS and OF\nwith motorized wheel (cm)",
    fontsize=fontsize_dict["label"],
)
sns.despine(ax=ax_f, offset=2, trim=False)


# ==============================================================================
# MIDDLE TIER: PANELS G - I (Three example cells on the treadmill)
# ==============================================================================
letters_mid = ["G", "H", "I"]
example_cells = [EXAMPLE_CELL, EXAMPLE_CELL2, EXAMPLE_CELL3]
n_rs_bins = 5

BAND_TOP = 0.585
BAND_H = 0.220
MATRIX_W = BAND_H / ASPECT / 2
MATRIX_TO_STACK = 0.048
CELL_GAP = 0.062
STACK_W = 0.108
STACK_GAP = 0.003
COMPONENT_COLORS = ("#0072B2", "#7F3C8D", "#009E73", "#E69F00", "#56B4E9")
example_colors = [COMPONENT_COLORS[2], COMPONENT_COLORS[0], COMPONENT_COLORS[1]]

band_bottom = BAND_TOP - BAND_H
stack_h = (BAND_H - (n_rs_bins - 1) * STACK_GAP) / n_rs_bins
block_w = MATRIX_W + MATRIX_TO_STACK + STACK_W
X0 = 0.048

range_kwargs = dict(
    log_range={"log_base": 2}, rs_bins=rs_bins, of_bins=of_bins, tick_dict=tick_dict
)

for iex, (cell_uid, letter) in enumerate(zip(example_cells, letters_mid)):
    block_x = X0 + iex * (block_w + CELL_GAP)
    stack_x = block_x + MATRIX_W + MATRIX_TO_STACK
    example_cell = neurons_df_treadmill[neurons_df_treadmill.roi_uid == cell_uid].iloc[0]
    roi = example_cell.roi

    # Panel lettering
    fig.text(
        block_x - 0.045,
        0.595,
        letter,
        fontsize=fontsize_dict.get("panel", 10),
        fontweight="bold",
        ha="left",
        va="top",
    )

    tav_df = []
    for trial, tseries in trials_df_tm.iterrows():
        ok = tseries.max_abs_rs2motor_diff_ratio_stim < max_abs_rs2motor_diff_ratio
        tav_df.append(
            dict(
                rs=tseries.RS_stim[ok].mean() * 100,
                of=np.degrees(tseries.OF_stim[ok].mean()),
                dff=tseries.dff_stim[:, roi].mean(),
            )
        )
    tav_df = pd.DataFrame(tav_df)

    axes_trials = [
        fig.add_axes(
            [
                stack_x,
                band_bottom + BAND_H - (i + 1) * stack_h - i * STACK_GAP,
                STACK_W,
                stack_h,
            ]
        )
        for i in range(n_rs_bins)
    ]
    for b_s, b_e, a in zip(rs_bins[2:], rs_bins[3:], axes_trials[::-1]):
        rsof_plots.plot_rsof_slice(
            a,
            b_s,
            b_e,
            tav_df,
            of_bins[1:],
            plot_trials=False,
            niter=10,
            color=example_colors[iex],
            fit_color=example_colors[iex],
            scatter_size=20,
            linewidth=1,
            capsize=1.5,
            markersize=3,
            fontsize_dict=fontsize_dict,
            clip_on=False,
        )
        a.set_xticklabels([])
        a.tick_params(axis="y", labelsize=fontsize_dict["tick"])
        a.spines[["top", "right"]].set_visible(False)

    def _panel_ymax(ax):
        vals = []
        for container in ax.containers:
            line, _, bars = container.lines
            vals.append(np.nanmax(line.get_ydata()))
            for bar in bars:
                segments = bar.get_segments()
                if len(segments):
                    vals.append(max(np.nanmax(seg[:, 1]) for seg in segments))
        for line in ax.lines:
            ydata = np.asarray(line.get_ydata(), dtype=float)
            if ydata.size > 100:
                vals.append(np.nanmax(ydata))
        return max(vals) if vals else np.nan

    ytop = np.floor(max(_panel_ymax(a) for a in axes_trials) / 0.1) * 0.1
    for iax, a in enumerate(axes_trials):
        a.set_ylim(-0.2, ytop * 1.2)
        a.set_yticks([])
        a.set_ylabel("")
        a.spines["left"].set_visible(False)
        if iax != (len(axes_trials) - 1):
            a.xaxis.set_visible(False)
            a.spines["bottom"].set_visible(False)

    bar = max(v for v in (0.1, 0.2, 0.5, 1, 2, 5) if v <= 0.75 * ytop)
    ax_bar = axes_trials[0]
    ax_bar.plot(
        [-0.03, -0.03],
        [0, bar],
        transform=ax_bar.get_yaxis_transform(),
        color="k",
        lw=1,
        clip_on=False,
    )
    ax_bar.text(
        -0.075,
        bar / 2,
        rf"{bar:g} $\Delta$F/F",
        transform=ax_bar.get_yaxis_transform(),
        rotation=90,
        ha="center",
        va="center",
        fontsize=fontsize_dict["tick"],
    )
    axes_trials[-1].set_xticks(
        [1, 10, 100, 1000],
        labels=["1", "10", "100", "1000"],
        fontsize=fontsize_dict["tick"],
    )
    axes_trials[-1].set_xlabel("Optic flow (°/s)", fontsize=fontsize_dict["label"])

    # RS/OF matrix heatmap
    ax_matrix = fig.add_axes([block_x, band_bottom, MATRIX_W, BAND_H])
    rsof_plots.plot_RS_OF_matrix(
        trials_df=trials_df_tm,
        roi=roi,
        is_closed_loop=1,
        max_abs_rs2motor_diff_ratio=max_abs_rs2motor_diff_ratio,
        ax=ax_matrix,
        fontsize_dict=fontsize_dict,
        vmin=0,
        vmax=ytop,
        cbar_width=0.01,
        title="",
        **range_kwargs,
    )


# ==============================================================================
# BOTTOM TIER: PANELS J - L (2D-Gaussian fit parameters, polar plot, mixture)
# ==============================================================================
fig.text(
    0.003,
    0.315,
    "J",
    fontsize=fontsize_dict.get("panel", 10),
    fontweight="bold",
    ha="left",
    va="top",
)
fig.text(
    0.248,
    0.315,
    "K",
    fontsize=fontsize_dict.get("panel", 10),
    fontweight="bold",
    ha="left",
    va="top",
)
fig.text(
    0.573,
    0.315,
    "L",
    fontsize=fontsize_dict.get("panel", 10),
    fontweight="bold",
    ha="left",
    va="top",
)

# --- Panel J: Fit schematic & formulas ---
fig.text(
    0.023,
    0.290,
    "Gaussian fit parameters",
    fontsize=fontsize_dict["title"],
    ha="left",
    va="top",
)

def draw_colored_run(fig, x, y, pieces, fontsize):
    renderer = fig.canvas.get_renderer()
    for string, color in pieces:
        txt = fig.text(
            x, y, string, color=color, fontsize=fontsize, ha="left", va="center"
        )
        txt.draw(renderer)
        x += txt.get_window_extent(renderer=renderer).width / fig.bbox.width
    return x

draw_colored_run(
    fig,
    0.018,
    0.245,
    [
        ("Elongation = ", "k"),
        (r"$\sigma_{major}$", SEMIMAJOR_COLOR),
        (" / ", "k"),
        (r"$\sigma_{minor}$", SEMIMINOR_COLOR),
    ],
    fontsize_dict["label"],
)
draw_colored_run(
    fig,
    0.055,
    0.215,
    [("Orientation ", "k"), (r"$\theta$", "#008080")],
    fontsize_dict["label"],
)

FIT_W = 0.065
fit_h = FIT_W * ASPECT * 2
FIT_X = 0.040
FIT_BOTTOM = 0.040
ax_fit = fig.add_axes([FIT_X, FIT_BOTTOM, FIT_W, fit_h])
example_cell3_obj = neurons_df_treadmill[neurons_df_treadmill.roi_uid == EXAMPLE_CELL4]
plot_g2d_fit_schematic(
    ax_fit,
    example_cell3_obj,
    roi=example_cell3_obj.roi.iloc[0],
    sfx=TA,
    min_sigma=MIN_SIGMA,
    fontsize_dict=fontsize_dict,
    mass_fraction=0.50,
    **range_kwargs,
)
ax_fit.set_title("")

# --- Panel K: Polar Scatter ---
POLAR_X, POLAR_BOTTOM, POLAR_W = 0.255, 0.035, 0.220
POLAR_H = POLAR_W * ASPECT
POLAR_IN_COLOR, POLAR_OUT_COLOR = "#444444", "darkred"
ax_polar = fig.add_axes([POLAR_X, POLAR_BOTTOM, POLAR_W, POLAR_H], projection="polar")
sc_kwargs = dict(s=20, alpha=0.5, linewidths=0, clip_on=False, zorder=3)
plot_angle_eccentricity_polar(
    ax_polar,
    ndf_polar.loc[elong_ok, f"g2d_theta{TA}"].astype(float),
    None,
    fontsize_dict,
    radial_scale="elongation",
    axis_ratio=elongation_polar[elong_ok],
    scale=0.5,
    rasterize_schematics="each",
    c=POLAR_IN_COLOR,
    **sc_kwargs,
)
ax_polar.scatter(
    np.radians(ndf_polar.loc[~elong_ok, f"g2d_theta{TA}"].astype(float)),
    np.clip(np.log2(elongation_polar[~elong_ok]), 0, None),
    c=POLAR_OUT_COLOR,
    **sc_kwargs,
)

# --- Panel L: Orientation Histogram & von Mises mixture ---
MAX_K = 5
N_ANGLE_BINS = 37
angle_edges = np.linspace(-45, 135, N_ANGLE_BINS + 1)
angle_bin_w = np.diff(angle_edges)[0]
angle_grid = np.linspace(-45, 135, 400)
VONMISES_COLOR = "#000000"

angles_deg = ndf_polar.loc[elong_ok, f"g2d_theta{TA}"].astype(float).dropna().to_numpy()

HIST_X, HIST_BOTTOM, HIST_W, HIST_H = 0.625, 0.045, 0.355, 0.245
ax_hist = fig.add_axes([HIST_X, HIST_BOTTOM, HIST_W, HIST_H])
ax_hist.hist(
    angles_deg, bins=angle_edges, color="lightgrey", edgecolor="k", linewidth=0.5
)

to_counts = len(angles_deg) * angle_bin_w

vm_results = vm.model_selection(
    np.radians(angles_deg), max_k=MAX_K, seed=42, verbose=False
)
k_vm = min(vm_results, key=lambda k: vm_results[k]["bic"])
r_vm = vm_results[k_vm]
pi_k, mu_k, kappa_k = (
    np.asarray(r_vm["pi_k"], dtype=float),
    np.asarray(r_vm["mu_k"], dtype=float),
    np.asarray(r_vm["kappa_k"], dtype=float),
)

def component_counts(idx):
    return (
        vm.axial_mixture_density(
            np.radians(angle_grid), pi_k[idx], mu_k[idx], kappa_k[idx]
        )
        / np.degrees(1)
        * to_counts
    )

ax_hist.plot(
    angle_grid,
    component_counts(slice(None)),
    color=VONMISES_COLOR,
    lw=2,
    zorder=3,
    label=f"von Mises mixture (k={k_vm})",
)

def wrap_axial_deg(deg):
    return (np.asarray(deg) + 45) % 180 - 45

mu_deg_vm = wrap_axial_deg(np.degrees(mu_k))
for rank, i_comp in enumerate(np.argsort(mu_deg_vm)):
    ax_hist.plot(
        angle_grid,
        component_counts([i_comp]),
        color=COMPONENT_COLORS[rank % len(COMPONENT_COLORS)],
        lw=1.5,
        ls="--",
        zorder=2,
        label=f"{mu_deg_vm[i_comp]:.0f}°: {100 * pi_k[i_comp]:.0f}%",
    )

ax_hist.set_xlim(-45, 135)
ax_hist.set_xticks([-45, 0, 45, 90, 135])
ax_hist.set_xlabel("Ellipse orientation (deg)", fontsize=fontsize_dict["label"])
ax_hist.set_ylabel("Number of cells", fontsize=fontsize_dict["label"])
ax_hist.tick_params(axis="both", labelsize=fontsize_dict["tick"], pad=1)
ax_hist.legend(fontsize=fontsize_dict["legend"], frameon=False, loc="upper left")
sns.despine(ax=ax_hist, offset=0, trim=True)

# Save only fig_depth_cells.svg as requested
style.savefig(SAVE_ROOT / "fig_depth_cells.svg", fig=fig, bbox_inches="tight", dpi=300)
print(f"Saved figure to {SAVE_ROOT / 'fig_depth_cells.svg'}")


In [ ]:
# Print N of neurons, sessions and mice for each panel:
# Pulled from the same objects the figure cell draws from, so these can't drift from
# what is actually plotted. Run after the figure cell (it needs `valid_neurons_treadmill`)
# and before the supplementary cells at the end, which rebind `ndf_polar`.
#
# Mice are counted from the session-name prefix ("PZAG17.3a_S20250402" -> "PZAG17.3a")
# rather than the `mouse` column, so these numbers match the legend cell below, which
# counts them the same way.
PANEL_B_TRIALS = np.arange(58, 62)  # mirrors `example_trials` in the figure cell


def panel_counts(df):
    """(n neurons, n sessions, n mice) for a slice of the population dataframe."""
    return (
        len(df),
        df.session.nunique(),
        df.session.str.split("_").str[0].nunique(),
    )


# Example-cell panels are all the one session, so their session/mouse counts are 1.
# `n_trials` is the full trial count of that session; panel B draws a window of it.
n_trials = len(trials_df_tm)
ONE = (1, 1, 1)

rows = [
    ("A", "Motorized wheel schematic", (None, None, None)),
    ("B", f"Protocol trace ({len(PANEL_B_TRIALS)}/{n_trials} trials)", (None, 1, 1)),
    ("C", f"Stimulus sampling ({n_trials} trials)", (None, 1, 1)),
    ("D", f"Depth tuning, free locomotion ({EXAMPLE_CELL_TREADMILL})", ONE),
    ("E", f"Free locomotion vs wheel matrices ({EXAMPLE_CELL_TREADMILL})", ONE),
    ("F", "RS/OF ratio vs preferred depth", panel_counts(valid_neurons_treadmill)),
    ("G", f"Example cell, running speed ({EXAMPLE_CELL})", ONE),
    ("H", f"Example cell, optic flow ({EXAMPLE_CELL2})", ONE),
    ("I", f"Example cell, RS/OF ratio ({EXAMPLE_CELL3})", ONE),
    ("J", f"Gaussian fit parameters ({EXAMPLE_CELL4})", ONE),
    ("K", "Elongation vs orientation, polar", panel_counts(ndf_polar)),
    ("L", "Orientation distribution", panel_counts(ndf_polar[elong_ok])),
]

# Width from the longest description, so the roi_uids never push the columns out.
w_what = max(len(what) for _, what, _ in rows)
header = f"{'':<2} {'panel':<{w_what}} {'neurons':>8} {'sessions':>9} {'mice':>5}"
print(header)
print("-" * len(header))
for letter, what, counts in rows:
    cells = [
        f"{v:>{w}}" if v is not None else f"{'-':>{w}}"
        for v, w in zip(counts, (8, 9, 5))
    ]
    print(f"{letter:<2} {what:<{w_what}} " + " ".join(cells))

# Context for the methods: what the panel K / L selection keeps out of everything loaded.
n_all, n_all_sessions, n_all_mice = panel_counts(neurons_df_treadmill)
print(
    f"\nloaded population: {n_all} neurons, {n_all_sessions} sessions, {n_all_mice} mice"
)
print(
    f"panel K (depth-tuned & significant g2d fit on the treadmill): "
    f"{len(ndf_polar)}/{n_all} ({len(ndf_polar) / n_all:.1%})"
)
print(
    f"panel L (of those, elongation > {ELONGATION_CUTOFF}): "
    f"{int(elong_ok.sum())}/{len(ndf_polar)} ({elong_ok.mean():.1%}), "
    f"{int((~elong_ok).sum())} below the cut"
)
print(
    f"panel F (depth-tuned & R2 above the empirical null): "
    f"{len(valid_neurons_treadmill)}/{n_all} ({len(valid_neurons_treadmill) / n_all:.1%})"
)
print("\nneurons per session")
print(ndf_polar.groupby("session").size().rename("panel K").to_string())

In [ ]:
# Hierarchical bootstrap correlation between preferred depth in free locomotion
# and expected depth (RS/OF ratio) on the motorized wheel (Panel F).
n_boots = 20000

xcol = ["expected_depth"]
ycol = ["preferred_depth_closedloop_crossval"]
np.random.seed(0)
valid_neurons_treadmill_stats = valid_neurons_treadmill.copy()
valid_neurons_treadmill_stats["expected_depth"] = (
    valid_neurons_treadmill_stats[f"preferred_RS_closedloop_crossval_g2d{TA}"]
    / valid_neurons_treadmill_stats[f"preferred_OF_closedloop_crossval_g2d{TA}"]
)

r, distribution = common_utils.hierarchical_bootstrap_stats(
    valid_neurons_treadmill_stats,
    n_boots,
    xcol,
    ["mouse", "session"],
    ycol=ycol,
    correlation=True,
)
pval = common_utils.calculate_pval_from_bootstrap(distribution, value=0)
print(
    f"Panel F Correlation check: Spearman r={r[0]:.3f}, p={pval:.3g}\n"
    "  -> tests whether the RS/OF ratio and preferred depth RANK together "
    "(monotonic association across neurons)."
)


# Figure legend

LAtex formatting and Science style

In [ ]:
# Figure legend, Science style. Numbers are pulled from the data loaded above rather
# than typed in, so the legend follows whatever the current fits/selection give.
import textwrap

FIG_NUM = "3"  # figure number in the manuscript

n_pop_k = len(valid_neurons_treadmill)
n_pop = len(ndf_polar)
n_elong = int(elong_ok.sum())
n_sessions = ndf_polar.session.nunique()
n_mice = ndf_polar.session.str.split("_").str[0].nunique()
components = ", ".join(
    rf"${mu_deg_vm[i]:.0f}^\circ$ ({100 * pi_k[i]:.0f}\%)"
    for i in np.argsort(mu_deg_vm)
)


def bf(s):
    """Bold, for the panel letters and the title sentence."""
    return r"\textbf{" + s + "}"


DFF = r"\Delta F/F"
ELONG = r"\sigma_{\mathrm{major}}/\sigma_{\mathrm{minor}}"

legend = [
    bf(f"Fig. {FIG_NUM}.") + " " + bf("V1 neurons are selective for visuomotor gains."),
    f"{bf('(A-C)')} Virtual reality setup and stimulus schematic for motorized wheel recordings.",
    f"{bf('(D and E)')} Virtual depth tuning during free locomotion {bf('(D)')} and responses of the "
    f"same neuron during free locomotion ({bf('E')}, left) and with the motorized wheel ({bf('E')}, right) "
    f"as a function of both running and optic flow speeds.",
    f"{bf('(F)')} Preferred depth tuning during free locomotion (mean of gaussian fit in {bf('D')}) "
    f"as function of the ratio between preferred running speed and optic flow with the motorized wheel "
    f"($N = {n_pop_k}$ neurons).",
    f"{bf('(G-I)')} Three example neurons, tuned to running speed {bf('(G)')}, to optic flow "
    f"{bf('(H)')} and to their ratio {bf('(I)')}. Left, trial-averaged ${DFF}$ against running speed "
    "and optic flow. Right, optic flow tuning within each running speed bin (Markers "
    "and errorbar indicate mean $\\pm$ bootstrapped 95\\% CI, line is a Gaussian fit. "
    "Triangles indicate the mean parameter of the fit).",
    f"{bf('(J)')} Description of fit parameters used.",
    f"{bf('(K)')} Elongation, ${ELONG}$, as a function of orientation, $\\theta$, for every depth-tuned "
    f"neuron with a significant fit ($n = {n_pop}$ neurons, ${n_sessions}$ sessions, "
    f"${n_mice}$ mice). Black, elongation $> {ELONGATION_CUTOFF}$ "
    f"(${n_elong}/{n_pop}$); dark red, below that cut-off.",
    f"{bf('(L)')} Distribution of orientations for neurons with elongated ellipses "
    f"($n = {n_elong}$, grey in {bf('K')}). Black, best axial von Mises mixture "
    f"($k = {k_vm}$); dashed, its components ({components}).",
]

# LaTeX source, ready to paste: one legend paragraph, one block per panel group.
# break_on_hyphens=False so a source line break never lands inside "cut-off",
# where LaTeX would turn it into a space.
print("\n".join(textwrap.fill(p, 88, break_on_hyphens=False) for p in legend))


# Export statistics for manuscript tracking
from v1_depth_map.stats import export_figure_stats

stats_fig3 = {
    "motorized_depth_tuned_neurons": {
        "raw": n_pop_k,
        "formatted": f"{n_pop_k}",
        "latex_macro": "statFigThreeMotorizedDepthNeurons",
        "description": "Depth-tuned neurons compared between free locomotion and motorized wheel (Fig 3F)",
    },
    "polar_n_neurons": {
        "raw": n_pop,
        "formatted": f"{n_pop}",
        "latex_macro": "statFigThreePolarNeurons",
        "description": "Neurons with significant 2D-Gaussian fit on motorized wheel (Fig 3K)",
    },
    "polar_n_sessions": {
        "raw": n_sessions,
        "formatted": f"{n_sessions}",
        "latex_macro": "statFigThreePolarSessions",
        "description": "Sessions in motorized wheel elongation analysis (Fig 3K)",
    },
    "polar_n_mice": {
        "raw": n_mice,
        "formatted": f"{n_mice}",
        "latex_macro": "statFigThreePolarMice",
        "description": "Mice in motorized wheel elongation analysis (Fig 3K)",
    },
    "elongation_cutoff_ratio": {
        "raw": f"{n_elong}/{n_pop}",
        "formatted": f"{n_elong}/{n_pop}",
        "latex_macro": "statFigThreeElongationCutoffRatio",
        "description": "Fraction of neurons with elongation ratio > 1.4 (Fig 3K)",
    },
    "elongation_n_neurons": {
        "raw": n_elong,
        "formatted": f"{n_elong}",
        "latex_macro": "statFigThreeElongatedNeurons",
        "description": "Neurons with elongated ellipses for orientation distribution (Fig 3L)",
    },
    "von_mises_k": {
        "raw": k_vm,
        "formatted": f"{k_vm}",
        "latex_macro": "statFigThreeVonMisesK",
        "description": "Best axial von Mises mixture number of components (Fig 3L)",
    },
    "von_mises_components": {
        "raw": components,
        "formatted": components,
        "latex_macro": "statFigThreeVonMisesComponents",
        "description": "Component centers and mixture weights for orientation distribution (Fig 3L)",
    },
}
export_figure_stats("figure3_depth_cells", stats_fig3, figure="fig3")


# Supplementaty analysis

In [ ]:
# Polar plot: four radial measures of the same tuning ellipses, same angles throughout.
# The first three differ only in how they compress the axis ratio a/b (a = semimajor,
# b = semiminor, so a/b >= 1 by construction); the fourth is not a shape measure at all.
#
#   flattening    1 - b/a              bounded [0, 1), roughly linear in shape
#   eccentricity  sqrt(1 - (b/a)^2)    bounded [0, 1) but saturates fast (2:1 -> 0.87)
#   log2(a/b)     unbounded, one unit per doubling of the ratio; what the paper panels use
#   sigma major   absolute size of the long axis, in ln(RS/OF) units (the g2d fit is done
#                 on natural-log RS and OF), i.e. e-folds -- 1 unit = 1.44 octaves
#
SFX = "_treadmill_trial_average_plateau"


mask = neurons_df_treadmill[f"rsof_test_rsq_closedloop_g2d{SFX}_sig"].fillna(
    False
) & neurons_df_treadmill["is_depth_neuron"].fillna(False)
ndf_polar = neurons_df_treadmill[mask].dropna(
    subset=[f"g2d_theta{SFX}", f"g2d_semimajor{SFX}", f"g2d_semiminor{SFX}"]
)
print(f"Number of neurons in polar plot: {len(ndf_polar)}/{len(neurons_df_treadmill)}")


theta = np.deg2rad(ndf_polar[f"g2d_theta{SFX}"].astype(float).to_numpy())
sigma_major = ndf_polar[f"g2d_semimajor{SFX}"].astype(float).to_numpy()
sigma_minor = ndf_polar[f"g2d_semiminor{SFX}"].astype(float).to_numpy()
# Clipped only to keep sqrt() and log2() real if a fit comes back with b marginally > a.
inv_ratio = np.clip(sigma_minor / sigma_major, 0, 1)
# Fits the orientation histogram / von Mises mixture drop as too round for their
# orientation to mean anything (elongation <= ELONGATION_CUTOFF, set in the cell that
# builds ndf_polar). Drawn in red on all four panels so the same cut can be judged
# against each radial measure.
below_cut = inv_ratio >= 1 / ELONGATION_CUTOFF
print(
    f"below the {ELONGATION_CUTOFF}:1 elongation cut (red): "
    f"{int(below_cut.sum())}/{len(below_cut)}"
)

LOG2_RIM = 3.0  # 8:1
SIGMA_RIM = 5


def to_rim(values, rim):
    """Park non-finite and over-rim values on the rim, and say how many there were."""
    finite = np.isfinite(values)
    n_rim = int((~finite).sum() + (finite & (values > rim)).sum())
    return np.where(finite, np.minimum(values, rim), rim), n_rim


log2_ratio, n_log2_rim = to_rim(-np.log2(np.maximum(inv_ratio, 1e-300)), LOG2_RIM)
sigma_parked, n_sigma_rim = to_rim(sigma_major, SIGMA_RIM)
n_unbounded = int((~(sigma_major <= 5)).sum())
print(
    f"semimajor: {n_unbounded}/{len(sigma_major)} fits wider than the sampled grid "
    f"(> 5 ln units), largest finite value "
    f"{np.nanmax(sigma_major[np.isfinite(sigma_major)]):.3g}"
)

RADII = [
    dict(label="Flattening  $1 - b/a$", radius=1 - inv_ratio, n_rim=0, rscale="linear"),
    dict(
        label=r"Eccentricity  $\sqrt{1 - (b/a)^2}$",
        radius=np.sqrt(1 - inv_ratio**2),
        n_rim=0,
        rscale="linear",
    ),
    dict(
        label=rf"$\log_2$ aspect ratio (clipped at {LOG2_RIM:.0f} = 8:1)",
        radius=log2_ratio,
        n_rim=n_log2_rim,
        rscale="linear",
    ),
    dict(
        label=rf"$\sigma_{{major}}$ (clipped at {SIGMA_RIM:.0f})",
        radius=sigma_parked,
        n_rim=n_sigma_rim,
        rscale="log",
    ),
]

fig, axes = plt.subplots(2, 2, subplot_kw=dict(projection="polar"), figsize=(5, 5))
scatter_kwargs = dict(s=10, alpha=0.5, linewidths=0, clip_on=False, zorder=3)
for ax, spec in zip(axes.flat, RADII):
    radius = spec["radius"]
    ax.scatter(theta[~below_cut], radius[~below_cut], color="black", **scatter_kwargs)
    ax.scatter(theta[below_cut], radius[below_cut], color="red", **scatter_kwargs)
    # Orientation is axial and wrapped to [-45, 135), so only that wedge can ever hold
    # points; the full circle would leave three quarters of each panel empty.
    ax.set_thetamin(-45)
    ax.set_thetamax(135)
    ax.set_thetagrids([-45, 0, 45, 90, 135])

    ax.set_title(spec["label"], fontsize=8, pad=14)
    ax.tick_params(labelsize=6)
    rim_note = f", {spec['n_rim']} parked on the rim" if spec["n_rim"] else ""
    print(
        f"{spec['label']:48s} {np.nanmin(radius):8.2f} .. {np.nanmax(radius):10.2f}"
        f"  (median {np.nanmedian(radius):6.2f}){rim_note}"
    )
fig.suptitle(f"Radial measure comparison, {SFX}", fontsize=9)
fig.tight_layout()

In [ ]:
# Standard error of each neuron's tuning-ellipse orientation, from the five fold fits.
#
# `fit_rs_of_tuning(k_folds=5)` stored a g2d fit per fold (`rsof_train_popt_*`, each on the
# 4/5 of trials of one train set) next to the full-data fit the figure uses, and nothing has
# ever read them. They are a delete-a-group jackknife: any two share 3/5 of their trials, so
# their raw spread understates the sampling variability of the full fit - a leave-one-group-
# out estimate moves only 1/(K-1) as far as that group would move the full estimate, and the
# jackknife factor puts that back (Kott 2001, "delete-a-group jackknife"):
#
#     Var = ((K-1)/K) * sum(delta_i**2)     ->     SE = 2 * RMS(delta)   for K = 5
#
# with delta_i the axial deviation of each fold angle from the folds' circular mean, wrapped
# to +/-90 deg.
#
# Three things to keep in view when reading the scatter:
#  - 5 replicates is 4 df, so one neuron's SE is only good to a factor ~1.5 (95% range of
#    SE_hat/SE_true is about 0.35-1.67). Trust the smoothed trend, not single points.
#  - wrapping caps the statistic: a completely unidentifiable neuron gives 2*90/sqrt(3) =
#    104 deg, not infinity, so the worst neurons look better than they are.
#  - the jackknife assumes a smooth statistic, and `get_gaussian_angle` flips by exactly
#    90 deg when a fold crosses elongation 1 - i.e. just below the cut.
from cottage_analysis.analysis import fit_gaussian_blob as fit_gb

TA_FOLD = "_treadmill_trial_average_plateau"  # same suffix as TA above
TRAIN_POPT = f"rsof_train_popt_closedloop_g2d{TA_FOLD}"
CUTOFF = 1.4  # the elongation cut, where the orientation stops being reproducible
LOG2_CLIP = (
    3.0  # 8:1; display only, ridge fits run to 1e30 - same rim as the polar panel
)
BIN_EDGES = np.array([0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, np.inf])
SMOOTH_BW = (
    0.3  # Gaussian kernel width, in log2-elongation units (a factor 1.23 in ratio)
)
SE_CEILING = 2 * 90 / np.sqrt(3)  # what the wrapped SE returns for random fold angles

# Rebuilt here rather than reusing `ndf_polar`: this cell sits at the end of the notebook
# and cell 10 rebinds that name with a slightly different expression, so whichever cell ran
# last would otherwise decide what this is computed on. Same mask as the cell that defines
# ELONGATION_CUTOFF.
ndf_fold = neurons_df_treadmill[
    neurons_df_treadmill[f"rsof_test_rsq_closedloop_g2d{TA_FOLD}_sig"]
    & neurons_df_treadmill["is_depth_neuron"].fillna(False)
].dropna(
    subset=[
        f"g2d_theta{TA_FOLD}",
        f"g2d_semimajor{TA_FOLD}",
        f"g2d_semiminor{TA_FOLD}",
    ]
)


def fold_thetas(popts):
    """Per-fold axial angles in degrees, or None if the fold fits are unusable."""
    if not isinstance(popts, (list, np.ndarray)) or len(popts) < 2:
        return None
    thetas = np.array(
        [fit_gb.get_gaussian_angle(np.asarray(p, dtype=float)) for p in popts]
    )
    return thetas if np.isfinite(thetas).all() else None


def axial_mean_deg(thetas):
    """Circular mean of axial angles, via the doubled angle, wrapped to [-45, 135)."""
    mean = np.degrees(np.angle(np.mean(np.exp(2j * np.radians(thetas))))) / 2
    return (mean + 45) % 180 - 45


def jackknife_se_deg(thetas):
    """Delete-a-group jackknife SE of the axial mean orientation, in degrees."""
    n_folds = len(thetas)
    dev = (thetas - axial_mean_deg(thetas) + 90) % 180 - 90
    return float(np.sqrt((n_folds - 1) / n_folds * np.sum(dev**2)))


rows, n_no_folds = [], 0
for _, neuron in ndf_fold.iterrows():
    thetas = fold_thetas(neuron[TRAIN_POPT])
    if thetas is None:
        n_no_folds += 1
        continue
    theta_full = float(neuron[f"g2d_theta{TA_FOLD}"])
    with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
        log2_elong = np.log2(
            float(neuron[f"g2d_semimajor{TA_FOLD}"])
            / float(neuron[f"g2d_semiminor{TA_FOLD}"])
        )
    rows.append(
        dict(
            roi_uid=neuron.roi_uid,
            session=neuron.session,
            n_folds=len(thetas),
            log2_elong=log2_elong,
            se_theta=jackknife_se_deg(thetas),
            # Consistency check: the full fit's angle against the mean of its own folds.
            full_vs_folds=float(
                abs((axial_mean_deg(thetas) - theta_full + 90) % 180 - 90)
            ),
        )
    )

fold_df = pd.DataFrame(rows)
# Binned on the clipped elongation, so ridge fits (log2 elongation inf) land in the top bin
# rather than falling outside the last interval and disappearing from the summary.
se_x = np.clip(fold_df.log2_elong.to_numpy(), 0, LOG2_CLIP)
fold_df["elong_bin"] = pd.cut(se_x, BIN_EDGES, right=False)
grouped = fold_df.groupby("elong_bin", observed=True)


def weighted_quantile(values, quantiles, weights):
    """Quantiles of `values` under `weights`, by interpolating the weighted CDF."""
    order = np.argsort(values)
    vals, wts = np.asarray(values)[order], np.asarray(weights)[order]
    cdf = (np.cumsum(wts) - 0.5 * wts) / np.sum(wts)
    return np.interp(quantiles, cdf, vals)


def kernel_quantiles(x, y, grid, bw, quantiles=(0.25, 0.5, 0.75), min_weight=3.0):
    """Local quantiles of y along x under a Gaussian kernel - a smooth running median.

    Median and quartiles rather than a running mean: the SE distribution is skewed and
    runs into its own 104 deg ceiling, which drags a mean around. The curve is left undrawn
    where too little supports it - `min_weight` is an effective-sample-size floor (weights
    sum to at most n), and the nearest-point check stops a dense clump from painting a
    confident line across a stretch of x that holds no neurons.
    """
    out = np.full((len(grid), len(quantiles)), np.nan)
    for i, x0 in enumerate(grid):
        w = np.exp(-0.5 * ((x - x0) / bw) ** 2)
        if w.sum() >= min_weight and np.min(np.abs(x - x0)) <= 2 * bw:
            out[i] = weighted_quantile(y, np.asarray(quantiles), w)
    return out


se_y = fold_df.se_theta.to_numpy()
se_grid = np.linspace(0, LOG2_CLIP, 200)
se_q = kernel_quantiles(se_x, se_y, se_grid, SMOOTH_BW)
# The resolution the cut buys: the local median SE at the cut itself.
se_at_cut = float(kernel_quantiles(se_x, se_y, np.log2([CUTOFF]), SMOOTH_BW)[0, 1])

fig_se, ax_se = plt.subplots(1, 1, figsize=(5.0, 4.0))
ax_se.scatter(se_x, se_y, s=12, alpha=0.5, linewidths=0, color="black", zorder=1)

# Smoothed local median rather than a binned one - a single neuron's SE has only 4 df, so
# the trend is the part worth reading, and bin edges put steps in it that the data has no
# opinion about. Note the smoother runs on the CLIPPED elongation, so everything at or
# beyond 8:1 piles onto the right-hand edge and the curve there describes that pile.
ax_se.fill_between(
    se_grid, se_q[:, 0], se_q[:, 2], color="darkred", alpha=0.15, lw=0, zorder=2
)
ax_se.plot(
    se_grid,
    se_q[:, 1],
    color="darkred",
    lw=1.5,
    zorder=3,
    label=f"local median (IQR), bw={SMOOTH_BW:g}",
)

# The cut, and the orientation SE it corresponds to.
ax_se.axvline(np.log2(CUTOFF), color="k", ls=":", lw=0.8, zorder=4)
ax_se.axhline(se_at_cut, color="k", ls=":", lw=0.8, zorder=4)
ax_se.plot(np.log2(CUTOFF), se_at_cut, "o", color="k", ms=4, zorder=5)
ax_se.text(
    np.log2(CUTOFF) + 0.04, 104, f"{CUTOFF:g}:1", fontsize=6, ha="left", va="top"
)
ax_se.text(
    LOG2_CLIP, se_at_cut + 1.5, f"{se_at_cut:.1f}°", fontsize=6, ha="right", va="bottom"
)

ax_se.axhline(SE_CEILING, color="grey", ls="--", lw=0.8, zorder=0)
ax_se.text(
    0.02,
    SE_CEILING - 2,
    "unidentifiable limit",
    fontsize=6,
    ha="left",
    va="top",
    color="grey",
)
ax_se.set_xticks([0, 1, 2, 3])
ax_se.set_xticklabels(["1:1", "2:1", "4:1", r"$\geq$8:1"])
ax_se.set_xlabel("Elongation of the full fit")
ax_se.set_ylabel("Jackknife SE of the orientation (deg)")
ax_se.set_ylim(-3, 110)
ax_se.set_yticks([0, 22.5, 45, 67.5, 90, 104])
ax_se.legend(fontsize=6, frameon=False, loc="upper right")
ax_se.spines[["top", "right"]].set_visible(False)
ax_se.set_title("Orientation SE from the 5 fold fits", fontsize=9)
fig_se.tight_layout()

# ---- Printout ------------------------------------------------------------------------
print(
    f"{len(fold_df)}/{len(ndf_fold)} neurons with usable fold fits "
    f"({n_no_folds} without), {fold_df.session.nunique()} sessions"
)
print(f"folds per neuron: {dict(fold_df.n_folds.value_counts().sort_index())}")
print(
    f"{int((fold_df.log2_elong > LOG2_CLIP).sum())} neurons above the {LOG2_CLIP:g} "
    f"(8:1) display clip, "
    f"{int((~np.isfinite(fold_df.log2_elong)).sum())} with non-finite elongation"
)
print(
    "full-fit angle vs its own folds' circular mean: median "
    f"{fold_df.full_vs_folds.median():.2f}°, mean {fold_df.full_vs_folds.mean():.2f}°, "
    f"max {fold_df.full_vs_folds.max():.1f}°"
)
print(
    f"neurons within 5° of the {SE_CEILING:.0f}° ceiling: "
    f"{int((fold_df.se_theta > SE_CEILING - 5).sum())}/{len(fold_df)}"
)

labels = [
    f"{2**i.left:.2g}+" if np.isinf(i.right) else f"{2**i.left:.2g}-{2**i.right:.2g}"
    for i in grouped.groups
]
print(f"\n{'elongation':>12} {'n':>4} {'med SE':>7} {'SE<=' + f'{se_at_cut:.0f}':>9}")
for (_, group), label in zip(grouped, labels):
    print(
        f"{label:>12} {len(group):4d} {group.se_theta.median():6.1f} "
        f"{(group.se_theta <= se_at_cut).mean() * 100:8.0f}%"
    )
print(
    f"{'all':>12} {len(fold_df):4d} {fold_df.se_theta.median():6.1f} "
    f"{(fold_df.se_theta <= se_at_cut).mean() * 100:8.0f}%"
)

# The local median read at a few elongations, the cut among them. `n within bw` is what the
# local estimate actually rests on; the bandwidth is an arbitrary choice, so the SE at the
# cut is also quoted at three of them - if those disagree, the curve is being read more
# finely than the data supports.
SE_AT_RATIOS = (1.25, CUTOFF, 1.5, 2.0, 3.0, 4.0)
se_at = kernel_quantiles(se_x, se_y, np.log2(np.array(SE_AT_RATIOS)), SMOOTH_BW)
print(f"\n{'elongation':>10} {'SE median':>10} {'SE IQR':>16} {'n within bw':>12}")
for ratio, (q25, med, q75) in zip(SE_AT_RATIOS, se_at):
    n_local = int((np.abs(se_x - np.log2(ratio)) <= SMOOTH_BW).sum())
    print(f"{ratio:8.2f}:1 {med:9.1f}° {f'{q25:.1f} - {q75:.1f}°':>16} {n_local:12d}")
bw_sens = [
    kernel_quantiles(se_x, se_y, np.log2([CUTOFF]), bw)[0, 1] for bw in (0.2, 0.3, 0.5)
]
print(
    f"\nSE at the {CUTOFF:g}:1 cut across bandwidths 0.2/0.3/0.5: "
    + " / ".join(f"{v:.1f}°" for v in bw_sens)
)